# Chapter 10A — Stage A: The **Identification Agent**

**Multi-Agent Analog EDA — Pedagogical Project (PhD-level notebook)**

---

## Problem statement

Analog designers and system architects often express **circuit intent** in **natural language**: performance targets, technology assumptions, loading environments, and architectural hints (e.g., *Miller compensation*, *two-stage OTA*).
Downstream agents (topology selection, sizing, layout, verification) require a **machine-auditable** representation.

**Goal of Stage A (this notebook).** Build a **self-contained identification module** that maps unstructured text to a structured **Component Inventory** JSON capturing:

- **Component types** (functional blocks implied by the text)
- **Specifications** (metrics, nominal targets, bounds)
- **Constraints** (min / max / inequality / typical)
- **Topology hints** and **technology context** (PDK, supply, load)

**Non-goal.** We deliberately avoid learned NER models; we use **regex + compositional rules** so the pipeline is **inspectable**, **deterministic**, and **course-friendly**.

### Learning objectives

1. Formalize hardware NER as **typed spans** over a domain ontology (COMPONENT, PARAMETER, VALUE, UNIT, CONSTRAINT, TOPOLOGY).
2. Implement a **rule-based tagger** with **overlap resolution** and **pattern libraries** for analog/mixed-signal lexicon.
3. Engineer a **specification parsing pipeline**: preprocess → extract → relate → normalize → validate.
4. Produce a **versioned JSON schema** (`component_inventory.v1`) consumable by later agents.
5. Treat **validation** as a first-class output: completeness, unit normalization, ambiguity flags, and **conservative inference** of missing-but-expected fields.

---


In [ ]:
from __future__ import annotations

import sys; sys.path.insert(0, '..')
from style_utils import (setup_3b1b_style, glow_line, glow_fill, styled_box,
                         styled_arrow, finish_plot, plotly_3b1b_layout,
                         BACKGROUND, SURFACE, TEXT, TEXT_DIM, GRID,
                         BLUE, TEAL, GREEN, YELLOW, GOLD, RED,
                         ROSE, PURPLE, CYAN, ORANGE, PALETTE)
setup_3b1b_style()

import json
import re
import unicodedata
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

pio.templates.default = "plotly_dark"

# Dark theme (matplotlib) — consistent with earlier course notebooks
DARK_BG = "#0d1117"
FG = "#c9d1d9"
MUTED = "#8b949e"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
AMBER = "#d29922"
RED = "#f85149"
PURPLE = "#bc8cff"
ORANGE = "#ffa657"

MPL_RC = {
    "figure.facecolor": DARK_BG,
    "axes.facecolor": DARK_BG,
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": FG,
    "text.color": FG,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "grid.color": "#21262d",
    "grid.alpha": 0.65,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "font.size": 11,
}
mpl.rcParams.update(MPL_RC)

print("Environment ready: regex-first identification agent (no ML weights required).")

## 2. Named entity recognition (NER) for hardware

We define a **closed label set** tailored to analog specification text:

| Label | Semantics (hardware) | Examples |
|------:|----------------------|----------|
| **TOPOLOGY** | Architectural / structural phrases | *two-stage*, *Miller-compensated*, *folded-cascode* |
| **COMPONENT** | Functional block or circuit class | *OTA*, *bandgap*, *LNA*, *PLL* |
| **PARAMETER** | Named figure-of-merit or electrical attribute | *gain*, *GBW*, *phase margin*, *PSRR* |
| **VALUE** | Numeric literal (possibly scientific) | `60`, `1.8`, `5e-12` |
| **UNIT** | Physical unit token | *dB*, *MHz*, *pF*, *mW*, *°* |
| **CONSTRAINT** | Relational / bound language | *at least*, *below*, `>`, *typical* |

### Why rule-based NER here?

Learned taggers excel at broad text; **analog specs** are **low-entropy templates** (metric tokens + numbers + units) interleaved with **idiosyncratic topology phrases**. A **transparent pattern library**:

- yields **explainable spans** (show your work for accreditation / debug),
- supports **hard guarantees** in pedagogy (repeatable outputs),
- interfaces cleanly with **relation extraction** (parameter↔value pairing via local windows).

---


In [ ]:
# --- Typed span representation -------------------------------------------------

@dataclass(frozen=True)
class Entity:
    label: str  # surface string
    kind: str  # COMPONENT | PARAMETER | ...
    start: int
    end: int

    @property
    def span(self) -> Tuple[int, int]:
        return (self.start, self.end)


PRIORITY = {
    "CONSTRAINT": 6,
    "TOPOLOGY": 5,
    "COMPONENT": 4,
    "PARAMETER": 3,
    "UNIT": 2,
    "VALUE": 1,
}


def _compile_named_patterns(
    mapping: Dict[str, Sequence[str]],
) -> List[Tuple[str, re.Pattern[str]]]:
    out: List[Tuple[str, re.Pattern[str]]] = []
    for kind, pats in mapping.items():
        for p in pats:
            out.append((kind, re.compile(p, re.IGNORECASE)))
    return out


HARDWARE_PATTERNS: Dict[str, Sequence[str]] = {
    "TOPOLOGY": [
        r"\btwo-?stage\b",
        r"\bsingle-?ended\b",
        r"\bfully-?differential\b",
        r"\bMiller[- ]compensated\b",
        r"\bfeed-?forward\b",
        r"\btelescopic\b",
        r"\bfolded[- ]cascode\b",
        r"\bclass[- ]AB\b",
        r"\binstrumentation\s+amplifier\b",
        r"\bcurrent-?mirror\s+load\b",
        r"\bdifferential\s+pair\b",
    ],
    "COMPONENT": [
        r"\bOTA\b",
        r"\bop[- ]?amp\b",
        r"\boperational\s+amplifier\b",
        r"\bLNA\b",
        r"\bmixer\b",
        r"\bVCO\b",
        r"\bPLL\b",
        r"\bDLL\b",
        r"\bADC\b",
        r"\bDAC\b",
        r"\bbandgap\b(?:\s+reference)?",
        r"\bBGR\b",
        r"\bLDO\b",
        r"\bcomparator\b",
        r"\bTIA\b",
        r"\bCDR\b",
    ],
    "PARAMETER": [
        r"\bDC\s+gain\b",
        r"\bgain\b(?![\w-])",
        r"\bGBW\b",
        r"\bunity[- ]gain(?:\s+bandwidth)?\b",
        r"\bbandwidth\b|\bBW\b(?![A-Za-z])",
        r"\bUGF\b",
        r"\bphase\s+margin\b|\bPM\b(?![A-Za-z])",
        r"\bPSRR\b",
        r"\bCMRR\b",
        r"\bICMR\b|\binput\s+common[- ]mode\s+range\b",
        r"\bslew\s+rate\b|\bSR\b(?![A-Za-z])",
        r"\bsettling\s+time\b",
        r"\boffset\b(?:\s+voltage)?",
        r"\bnoise\b",
        r"\bSNR\b",
        r"\bSFDR\b",
        r"\bENOB\b",
        r"\bTHD\b",
        r"\bpower(?:\s+consumption)?\b",
        r"\bsupply\b(?:\s+voltage)?",
        r"\bVDD\b|\bVSS\b",
        r"\bload\s+capacitance\b|\bC_L\b|\bCL\b",
        r"\boutput\s+swing\b",
    ],
    "UNIT": [
        r"\bdB\b",
        r"\bGHz\b|\bMHz\b|\bkHz\b|\bHz\b",
        r"\bmV\b|\bV\b",
        r"\bµW\b|\buW\b|\bmW\b|\bW\b",
        r"\bpF\b|\bnF\b|\bµF\b|\buF\b|\bF\b",
        r"°|\bdeg(?:rees)?\b",
        r"\bps\b|\bns\b|\bµs\b|\bus\b|\bms\b|\bs\b",
        r"\bkΩ\b|\bKohm\b|\bohm\b|\bΩ\b|\bOhms?\b",
        # Avoid tagging the English article "a" as amperes: require a numeric literal before A/mA/µA.
        r"(?<=\d)\s*mA\b",
        r"(?<=\d)\s*µA\b",
        r"(?<=\d)\s*uA\b",
        r"(?<=\d)\s*A\b",
    ],
    "CONSTRAINT": [
        r"\bat\s+least\b",
        r"\bat\s+most\b",
        r"\bno\s+more\s+than\b",
        r"\bno\s+less\s+than\b",
        r"\bbelow\b",
        r"\babove\b",
        r"\bgreater\s+than\b",
        r"\bless\s+than\b",
        r"\bbetween\b",
        r"\btypical\b|\btyp\.?\b",
        r"\bnominal\b",
        r"\bmin(?:imum)?\b",
        r"\bmax(?:imum)?\b",
        r"\bapproximately\b|\babout\b|\baround\b",
        r">=|<=|≥|≤|~|±|>|<",
    ],
    "VALUE": [
        r"(?<![A-Za-z0-9_])(?:\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)(?![A-Za-z0-9_])",
    ],
}


class HardwareNER:
    "Rule-based span tagger with greedy overlap resolution by (priority, length, start)."

    def __init__(self, mapping: Dict[str, Sequence[str]] = HARDWARE_PATTERNS) -> None:
        self._rules = _compile_named_patterns(mapping)

    def extract(self, text: str) -> List[Entity]:
        candidates: List[Entity] = []
        for kind, rx in self._rules:
            for m in rx.finditer(text):
                candidates.append(Entity(m.group(0), kind, m.start(), m.end()))

        # Prefer earlier start, then higher ontology priority, then longer span
        candidates.sort(
            key=lambda e: (e.start, -PRIORITY[e.kind], -(e.end - e.start), e.kind)
        )
        chosen: List[Entity] = []
        for e in candidates:
            overlap = [c for c in chosen if not (e.end <= c.start or e.start >= c.end)]
            if not overlap:
                chosen.append(e)
                continue
            beats_all = True
            losers: List[Entity] = []
            for c in overlap:
                pe, pc = PRIORITY[e.kind], PRIORITY[c.kind]
                le, lc = e.end - e.start, c.end - c.start
                if pe > pc or (pe == pc and le > lc) or (pe == pc and le == lc and e.kind < c.kind):
                    losers.append(c)
                else:
                    beats_all = False
                    break
            if beats_all:
                chosen = [c for c in chosen if c not in losers]
                chosen.append(e)

        chosen.sort(key=lambda x: (x.start, x.end))
        return chosen


# Quick smoke test on hardware-like prose
_demo_text = (
    "Target: two-stage Miller-compensated OTA, GBW ~ 100 MHz, phase margin > 60°, "
    "PSRR at least 60 dB at 1 kHz."
)
_demo_ner = HardwareNER()
_demo_entities = _demo_ner.extract(_demo_text)
print("Demo sentence:\n ", _demo_text)
print("\nExtracted entities (span → kind: text):")
for ent in _demo_entities:
    print(f"  [{ent.start:3d}:{ent.end:3d}] {ent.kind:12s} {ent.label!r}")


## 3. Specification parsing pipeline

We implement a **sequential pipeline** $x \mapsto \mathcal{E}(x) \mapsto \mathcal{R}(\mathcal{E}) \mapsto J$, where:

1. **Text preprocessing** normalizes unicode, hyphenation, and whitespace; it also expands a *small* synonym table (GBW ↔ unity-gain bandwidth).
2. **Entity extraction** produces typed spans $\mathcal{E}$ (Section 2).
3. **Relation extraction** attaches **values** to **parameters** using **clause regexes** first, then **bidirectional proximity windows** as a fallback. When a numeric span is already owned by a high-confidence clause, we **suppress** redundant proximity links (clean specs often show `relation_hints: []`).
4. **Constraint identification** maps relational language to a discrete bound type (`min`, `max`, `approx`, `ineq_gt`, ...).
5. **JSON schema generation** emits `component_inventory` v1 with explicit **validation** and **ambiguity** slots.

The design choice mirrors classical **information extraction** pipelines in NLP, but the **ontology** is **EDA-native**.

---


In [ ]:
# --- Preprocessing ------------------------------------------------------------

SYNONYM_MAP = [
    (re.compile(r"\bugf\b", re.I), "unity gain bandwidth"),
    # Keep the token "GBW" — expanding to "gain bandwidth product" duplicates the PARAMETER tag "gain".
    (re.compile(r"\bpm\b(?![a-z])", re.I), "phase margin"),
]


def preprocess_spec(text: str) -> str:
    t = unicodedata.normalize("NFKC", text)
    t = t.replace("μ", "µ").replace("​", "")  # thin space / weird unicode
    t = re.sub(r"\s+", " ", t).strip()
    for rx, rep in SYNONYM_MAP:
        t = rx.sub(rep, t)
    return t


# --- Clause-level relation patterns (high precision) -------------------------

FLOAT = r"(?P<val>\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"
UNIT_DB = r"(?P<unit>dB)"
UNIT_FREQ = r"(?P<unit>GHz|MHz|kHz|Hz)"
UNIT_CAP = r"(?P<unit>pF|nF|µF|uF|F)"
UNIT_PWR = r"(?P<unit>mW|µW|uW|W)"
UNIT_VOLT = r"(?P<unit>mV|V)"
DEG = r"(?P<unit>°|deg(?:rees)?)"


def _bound_from_prefix(s: Optional[str]) -> str:
    if not s:
        return "target"
    s2 = s.strip().lower()
    if s2 in {">", "greater than"}:
        return "gt"
    if s2 in {"<", "less than", "below", "under"}:
        return "lt"
    if s2 in {">=", "≥", "at least", "no less than"}:
        return "min"
    if s2 in {"<=", "≤", "at most", "no more than"}:
        return "max"
    if s2 in {"~", "approximately", "about", "around", "typical", "typ.", "nominal"}:
        return "typ"
    return "target"


CLAUSES: List[Tuple[str, re.Pattern[str]]] = [
    (
        "dc_gain",
        re.compile(
            rf"(?P<pre>(?:at\s+least|at\s+most|greater\s+than|less\s+than|>=|<=|≥|≤|>|<|~|approximately|around|typical|typ\.?)?\s*)"
            rf"{FLOAT}\s*{UNIT_DB}\s+(?:of\s+)?(?:(?:dc\s+)?gain|gain)\b",
            re.I,
        ),
    ),
    (
        "dc_gain",
        re.compile(
            rf"(?P<pre>(?:at\s+least|at\s+most|greater\s+than|less\s+than|>=|<=|≥|≤|>|<|~)?\s*)"
            rf"(?:(?:dc\s+)?gain|gain)\s*(?:of|:)?\s*{FLOAT}\s*{UNIT_DB}",
            re.I,
        ),
    ),
    (
        "gain_bandwidth_hz",
        re.compile(
            rf"(?P<pre>(?:>|>=|≥|<|<=|≤|~|approximately|around|at\s+least|at\s+most)?\s*)"
            rf"{FLOAT}\s*{UNIT_FREQ}\s+(?:GBW|gain\s+bandwidth\s+product|unity\s+gain\s+bandwidth)\b",
            re.I,
        ),
    ),
    (
        "gain_bandwidth_hz",
        re.compile(
            rf"(?P<pre>(?:>|>=|≥|<|<=|≤|~)?\s*)"
            rf"(?:GBW|gain\s+bandwidth\s+product|unity\s+gain\s+bandwidth)\s*(?:of|:|@)?\s*{FLOAT}\s*{UNIT_FREQ}",
            re.I,
        ),
    ),
    (
        "gain_bandwidth_hz",
        re.compile(
            rf"(?P<pre>)\b(?:bandwidth|BW)\s+{FLOAT}\s*{UNIT_FREQ}\b",
            re.I,
        ),
    ),
    (
        "phase_margin_deg",
        re.compile(
            rf"(?P<pre>(?:>|>=|≥|<|<=|≤|~|at\s+least|at\s+most|greater\s+than|less\s+than)?\s*)"
            rf"{FLOAT}\s*{DEG}\s+(?:phase\s+margin)\b",
            re.I,
        ),
    ),
    (
        "phase_margin_deg",
        re.compile(
            rf"(?:phase\s+margin)\s*(?P<pre>(?:>|>=|≥|<|<=|≤|~|at\s+least|at\s+most|greater\s+than|less\s+than)?)\s*{FLOAT}\s*{DEG}?",
            re.I,
        ),
    ),
    (
        "power_w",
        re.compile(
            rf"(?P<pre>(?:below|under|less\s+than|at\s+most|no\s+more\s+than|>|>=|<|<=|≤)?\s*)"
            rf"(?:power(?:\s+consumption)?)\s+(?:of|:)?\s*{FLOAT}\s*{UNIT_PWR}",
            re.I,
        ),
    ),
    (
        "power_w",
        re.compile(
            rf"(?:power(?:\s+consumption)?)\s+(?P<pre>below|under|less\s+than|at\s+most|no\s+more\s+than|greater\s+than|at\s+least)\s+{FLOAT}\s*{UNIT_PWR}",
            re.I,
        ),
    ),
    (
        "power_w",
        re.compile(
            rf"(?P<pre>(?:below|under|less\s+than|at\s+most|no\s+more\s+than)?\s*)"
            rf"{FLOAT}\s*{UNIT_PWR}\s+(?:power(?:\s+consumption)?)\b",
            re.I,
        ),
    ),
    (
        "supply_voltage_v",
        re.compile(
            rf"(?:SKY130|TSMC\d+[a-z]*|GF\d+[a-z]*)\s+(?P<pre>){FLOAT}\s*{UNIT_VOLT}\s+supply",
            re.I,
        ),
    ),
    (
        "supply_voltage_v",
        re.compile(
            rf"(?P<pre>)\b(?:using|with)\s+\S+\s+{FLOAT}\s*{UNIT_VOLT}\s+supply\b",
            re.I,
        ),
    ),
    (
        "load_capacitance_f",
        re.compile(
            rf"(?P<pre>)(?:driving|drive)\s+{FLOAT}\s*{UNIT_CAP}\s+load",
            re.I,
        ),
    ),
    (
        "load_capacitance_f",
        re.compile(
            rf"(?P<pre>)(?:load)\s+(?:of|:)?\s*{FLOAT}\s*{UNIT_CAP}",
            re.I,
        ),
    ),
]


FREQ_TO_HZ = {"Hz": 1.0, "kHz": 1e3, "MHz": 1e6, "GHz": 1e9}
CAP_TO_F = {"F": 1.0, "uF": 1e-6, "µF": 1e-6, "nF": 1e-9, "pF": 1e-12}
PWR_TO_W = {"W": 1.0, "mW": 1e-3, "uW": 1e-6, "µW": 1e-6}
VOLT_TO_V = {"V": 1.0, "mV": 1e-3}


def normalize_to_si(metric_key: str, val: float, unit: str) -> Dict[str, Any]:
    unit_norm = unit.replace("u", "µ")
    if metric_key.endswith("_hz"):
        mul = FREQ_TO_HZ.get(unit, float("nan"))
        return {"value_si": val * mul, "si_unit": "Hz"}
    if metric_key.endswith("_f"):
        mul = CAP_TO_F.get(unit_norm, float("nan"))
        return {"value_si": val * mul, "si_unit": "F"}
    if metric_key.endswith("_w"):
        mul = PWR_TO_W.get(unit_norm, float("nan"))
        return {"value_si": val * mul, "si_unit": "W"}
    if metric_key.endswith("_v"):
        mul = VOLT_TO_V.get(unit_norm, float("nan"))
        return {"value_si": val * mul, "si_unit": "V"}
    if metric_key.endswith("_deg"):
        return {"value_si": val, "si_unit": "deg"}
    if metric_key.endswith("gain") or metric_key == "dc_gain":
        return {"value_si": val, "si_unit": "dB"}
    return {"value_si": val, "si_unit": unit_norm}


def extract_metric_clauses(text: str) -> List[Dict[str, Any]]:
    hits: List[Dict[str, Any]] = []
    seen_spans: set[Tuple[int, int, str]] = set()
    for key, rx in CLAUSES:
        for m in rx.finditer(text):
            span = (m.start(), m.end(), key)
            if span in seen_spans:
                continue
            seen_spans.add(span)
            gd = m.groupdict()
            pre = gd.get("pre")
            val = float(gd["val"])
            unit = gd.get("unit") or ""
            if unit.lower() in {"deg", "degrees"}:
                unit_disp = "deg"
            elif unit == "°":
                unit_disp = "deg"
            else:
                unit_disp = unit
            bound = _bound_from_prefix(pre)
            hits.append(
                {
                    "metric": key,
                    "raw_text": m.group(0),
                    "span": [m.start(), m.end()],
                    "value": val,
                    "unit": unit_disp,
                    "bound_type": bound,
                    "normalized": normalize_to_si(key, val, unit_disp),
                }
            )
    hits.sort(key=lambda h: h["span"][0])
    return hits


def relation_hints_from_entities(
    text: str,
    entities: Sequence[Entity],
    window: int = 40,
    covered_spans: Optional[Sequence[Tuple[int, int]]] = None,
) -> List[Dict[str, Any]]:
    # Heuristic fallback: link PARAMETER ↔ VALUE ±UNIT within a window (both directions).
    covered = list(covered_spans or [])

    def span_covered(s: int, e: int) -> bool:
        return any(not (e <= a or s >= b) for a, b in covered)

    def nearest_unit(v: Entity, units: Sequence[Entity]) -> Optional[Entity]:
        after = [u for u in units if v.end <= u.start <= v.end + 10]
        if after:
            return min(after, key=lambda u: u.start - v.end)
        before = [u for u in units if u.end <= v.start and v.start - u.end <= 10]
        if before:
            return min(before, key=lambda u: v.start - u.end)
        return None

    by_start = sorted(entities, key=lambda e: e.start)
    params = [e for e in by_start if e.kind == "PARAMETER"]
    values = [e for e in by_start if e.kind == "VALUE"]
    units = [e for e in by_start if e.kind == "UNIT"]
    hints: List[Dict[str, Any]] = []
    for p in params:
        best: Optional[Tuple[Entity, Optional[Entity], int]] = None
        for v in values:
            if span_covered(v.start, v.end):
                continue
            if v.end <= p.start:
                dist = p.start - v.end
            elif v.start >= p.end:
                dist = v.start - p.end
            else:
                continue
            if dist > window:
                continue
            u = nearest_unit(v, units)
            score = dist + (0 if u else 4)
            if best is None or score < best[2]:
                best = (v, u, score)
        if best:
            v, u, _ = best
            hints.append(
                {
                    "parameter_span": [p.start, p.end],
                    "parameter_text": text[p.start : p.end],
                    "value_span": [v.start, v.end],
                    "value_text": v.label,
                    "unit_span": ([u.start, u.end] if u else None),
                    "unit_text": (u.label if u else None),
                    "method": "proximity_bidir_window",
                }
            )
    return hints


print("Clause extractor loaded:", len(CLAUSES), "patterns")


## 4. Component inventory JSON (schema v1)

The **Component Inventory** is the Stage-A contract for downstream agents. It partitions knowledge into:

- **`topology_hints`**: non-netlist structural language.
- **`functional_blocks`**: COMPONENTS with optional roles.
- **`technology`**: PDK tokens, inferred nodes, supplies.
- **`performance_specifications`**: structured metrics with **SI-normalized** fields.
- **`loading_conditions`**: explicit environmental loads.
- **`inferred_missing`**: *conservative* suggestions when common specs are absent.
- **`validation`**: completeness / ambiguity / normalization notes.

This is intentionally **verbose** for pedagogy; production systems might compress or hash redundant spans.

---


In [ ]:
PDK_RX = re.compile(r"\b(SKY130|TSMC\d+[a-z]*|GF\d+[a-z]*|IBM\d+[a-z]*)\b", re.I)

# Machine-readable contract (validate externally with `jsonschema` if installed)
COMPONENT_INVENTORY_JSON_SCHEMA_V1: Dict[str, Any] = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "AnalogIdentification_ComponentInventory_v1",
    "type": "object",
    "required": ["schema", "version", "raw_specification"],
    "properties": {
        "schema": {"const": "component_inventory"},
        "version": {"type": "string"},
        "raw_specification": {"type": "string"},
        "normalized_text": {"type": "string"},
        "topology_hints": {"type": "array", "items": {"type": "string"}},
        "functional_blocks": {"type": "array"},
        "inferred_passives": {"type": "array"},
        "technology": {"type": "object"},
        "performance_specifications": {"type": "array"},
        "loading_conditions": {"type": "array"},
        "relation_hints": {"type": "array"},
        "extracted_entities": {"type": "array"},
        "validation": {"type": "object"},
    },
}


def extract_pdk(text: str) -> Optional[str]:
    m = PDK_RX.search(text)
    return m.group(1).upper() if m else None


def infer_functional_blocks(
    text: str, entities: Sequence[Entity]
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    blocks: List[Dict[str, Any]] = []
    comp_labels = {e.label.lower() for e in entities if e.kind == "COMPONENT"}
    if "ota" in text.lower() or any("ota" in c for c in comp_labels):
        blocks.append({"kind": "OTA", "confidence": "high", "evidence": "text_or_span"})
    if re.search(r"\bbandgap\b|\bBGR\b", text, re.I):
        blocks.append({"kind": "BANDGAP_REFERENCE", "confidence": "medium", "evidence": "keyword"})
    if re.search(r"\bLDO\b", text, re.I):
        blocks.append({"kind": "LDO", "confidence": "medium", "evidence": "keyword"})
    if re.search(r"\bCDR\b|\bFFE\b|\bCTLE\b|\bPAM4\b", text, re.I):
        blocks.append({"kind": "WIRELINE_RX_SLICE", "confidence": "medium", "evidence": "keyword"})
    inferred_passive: List[Dict[str, Any]] = []
    if re.search(r"Miller[- ]compensated", text, re.I):
        inferred_passive.append(
            {
                "kind": "MILLER_COMPENSATION_NETWORK",
                "note": "Topology phrase implies Cc (and often nulling R) between internal nodes — netlist not synthesized at Stage A.",
            }
        )
    return blocks, inferred_passive


def build_component_inventory(raw: str) -> Dict[str, Any]:
    text = preprocess_spec(raw)
    ner = HardwareNER()
    entities = ner.extract(text)
    metrics = extract_metric_clauses(text)
    pdk = extract_pdk(text)

    topo = sorted({e.label for e in entities if e.kind == "TOPOLOGY"})
    comps, inferred_passive = infer_functional_blocks(text, entities)

    # Technology section
    supply_entries = [m for m in metrics if m["metric"] == "supply_voltage_v"]
    vdd = supply_entries[0]["value"] if supply_entries else None

    loads = [m for m in metrics if m["metric"] == "load_capacitance_f"]

    perf = [m for m in metrics if m["metric"] not in {"supply_voltage_v", "load_capacitance_f"}]
    covered_metric_spans = [tuple(m["span"]) for m in metrics]
    relation_hints = relation_hints_from_entities(text, entities, covered_spans=covered_metric_spans)

    inventory: Dict[str, Any] = {
        "schema": "component_inventory",
        "version": "1.0",
        "raw_specification": raw,
        "normalized_text": text,
        "topology_hints": topo,
        "functional_blocks": comps,
        "inferred_passives": inferred_passive,
        "technology": {
            "pdk_token": pdk,
            "nominal_supply_v": vdd,
            "inferred_ground_reference": "VSS" if vdd is not None else None,
        },
        "performance_specifications": perf,
        "loading_conditions": loads,
        "relation_hints": relation_hints,
        "extracted_entities": [
            {
                "label": e.label,
                "kind": e.kind,
                "span": [e.start, e.end],
            }
            for e in entities
        ],
    }
    return inventory


PRIMARY_SPEC = (
    "Design a two-stage Miller-compensated OTA with at least 60dB gain, 100MHz GBW, "
    "phase margin >60°, power consumption below 1mW, using SKY130 1.8V supply, driving 5pF load"
)

inv0 = build_component_inventory(PRIMARY_SPEC)
print("JSON Schema (v1) top-level keys:", list(COMPONENT_INVENTORY_JSON_SCHEMA_V1["properties"].keys()))
print(json.dumps(inv0, indent=2)[:4000])
print("\n... [truncated for display; full JSON returned in `inv0`]")


## 5. Validation, normalization, ambiguity, and missing-spec inference

**Validation** is not an afterthought: for agentic EDA, it is the **gating function** that prevents unsafe downstream compilation.

We implement:

1. **Completeness checking** against a **metric checklist** for the inferred functional class (OTA example).
2. **Unit normalization** (already stored under `normalized` for metrics).
3. **Ambiguity resolution** flags: duplicate metrics, conflicting bounds, unclear units.
4. **Missing-spec inference**: conservative suggestions (not auto-filled targets) — e.g., if an OTA lacks **slew rate** or **ICMR**, we emit *information requests* rather than hallucinating numbers.

---


In [ ]:
OTA_EXPECTED_METRICS = {
    "dc_gain",
    "gain_bandwidth_hz",
    "phase_margin_deg",
    "power_w",
    "supply_voltage_v",
    "load_capacitance_f",
}


def detect_ambiguities(inv: Dict[str, Any]) -> List[Dict[str, Any]]:
    issues: List[Dict[str, Any]] = []
    per_metric: Dict[str, List[Dict[str, Any]]] = {}
    for m in inv.get("performance_specifications", []):
        per_metric.setdefault(m["metric"], []).append(m)
    for k, lst in per_metric.items():
        if len(lst) > 1:
            issues.append(
                {
                    "type": "duplicate_metric",
                    "metric": k,
                    "count": len(lst),
                    "note": "Multiple clauses mapped to the same metric key — review spans.",
                }
            )
    # Numeric sanity
    for m in inv.get("performance_specifications", []):
        if m["metric"] == "phase_margin_deg" and not (0 < m["value"] < 90):
            issues.append(
                {
                    "type": "unlikely_value",
                    "metric": m["metric"],
                    "value": m["value"],
                    "note": "Phase margin usually 45–85° for intended closed-loop stability.",
                }
            )
    return issues


def completeness_for_ota(inv: Dict[str, Any]) -> Dict[str, Any]:
    present = {m["metric"] for m in inv.get("performance_specifications", [])}
    present |= {m["metric"] for m in inv.get("loading_conditions", [])}
    if inv.get("technology", {}).get("nominal_supply_v") is not None:
        present.add("supply_voltage_v")
    missing = sorted(OTA_EXPECTED_METRICS - present)
    score = 1.0 - (len(missing) / max(1, len(OTA_EXPECTED_METRICS)))
    return {"missing_metrics": missing, "completeness_score": round(score, 3)}


def infer_missing_specs(inv: Dict[str, Any]) -> List[Dict[str, Any]]:
    suggestions: List[Dict[str, Any]] = []
    blocks = {b.get("kind") for b in inv.get("functional_blocks", [])}
    if "OTA" in blocks:
        if not any(m["metric"] == "power_w" for m in inv.get("performance_specifications", [])):
            suggestions.append(
                {
                    "field": "power_w",
                    "severity": "high",
                    "action": "request_clarification",
                    "reason": "Power budget strongly constrains device sizes and bias network.",
                }
            )
        if not any("slew" in e["label"].lower() for e in inv.get("extracted_entities", [])):
            suggestions.append(
                {
                    "field": "slew_rate",
                    "severity": "medium",
                    "action": "optional_follow_up",
                    "reason": "Large-signal specs often independent of small-signal GBW/PM.",
                }
            )
        if not any("cmrr" in e["label"].lower() for e in inv.get("extracted_entities", [])):
            suggestions.append(
                {
                    "field": "CMRR",
                    "severity": "low",
                    "action": "optional_follow_up",
                    "reason": "Differential OTAs frequently specify CMRR/PSRR across frequency.",
                }
            )
    return suggestions


def validate_inventory(inv: Dict[str, Any]) -> Dict[str, Any]:
    report = {
        "completeness": completeness_for_ota(inv),
        "ambiguities": detect_ambiguities(inv),
        "unit_normalization_ok": all(
            isinstance(m.get("normalized", {}).get("value_si"), (int, float))
            and np.isfinite(m["normalized"]["value_si"])
            for m in inv.get("performance_specifications", []) + inv.get("loading_conditions", [])
        ),
        "inferred_missing": infer_missing_specs(inv),
    }
    return report


inv0["validation"] = validate_inventory(inv0)
print(json.dumps(inv0["validation"], indent=2))


## 6. Interactive demo — multiple specs, side-by-side rendering, entity visualization

The cell below runs **three** diverse examples (OTA, LDO-ish, wireline front-end language) and:

- prints a compact **input → JSON** table via HTML for notebook readability,
- builds a **span painting** figure: each entity type is a row; spans are drawn as horizontal bars over character indices (dark theme).

> **Note.** This is a *pedagogical* extractor: it will miss creative phrasing. The engineering takeaway is how to **iterate pattern libraries** and **validation gates** — the same interfaces you'd attach to an LLM for hybrid extraction later.

---


In [ ]:
EXAMPLES: Dict[str, str] = {
    "OTA (primary)": PRIMARY_SPEC,
    "Bandgap + noise": (
        "Generate a CMOS bandgap reference in SKY130 with nominal 1.2V output, "
        "temperature coefficient below 20ppm/C, and integrated noise under 100µV RMS from 0.1Hz to 10Hz."
    ),
    "Wireline CDR": (
        "Architect a 56Gbps PAM4 wireline RX slice with CTLE, 4-tap FFE, and a bang-bang CDR; "
        "target BER below 1e-12 post-FEC, supply 0.9V digital / 1.2V analog domains."
    ),
}


def inventory_to_compact_json(inv: Dict[str, Any]) -> str:
    slim = {
        "topology_hints": inv.get("topology_hints"),
        "functional_blocks": inv.get("functional_blocks"),
        "technology": inv.get("technology"),
        "performance_specifications": inv.get("performance_specifications"),
        "loading_conditions": inv.get("loading_conditions"),
        "relation_hints": inv.get("relation_hints"),
        "validation": inv.get("validation"),
    }
    return json.dumps(slim, indent=2)


def side_by_side_table(rows: List[Tuple[str, str, str]]) -> HTML:
    trs = []
    for title, spec, js in rows:
        trs.append(
            "<tr>"
            f"<td style='vertical-align:top;width:18%;color:#8b949e'>{title}</td>"
            f"<td style='vertical-align:top;width:41%;white-space:pre-wrap;font-family:ui-monospace,monospace;font-size:11px'>{spec}</td>"
            f"<td style='vertical-align:top;width:41%;white-space:pre-wrap;font-family:ui-monospace,monospace;font-size:11px'>{js}</td>"
            "</tr>"
        )
    html = (
        "<table style='width:100%;border-collapse:collapse;background:#0d1117;color:#c9d1d9'>"
        "<thead><tr>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #30363d'>Example</th>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #30363d'>Input text</th>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #30363d'>Compact JSON</th>"
        "</tr></thead><tbody>"
        + "".join(trs)
        + "</tbody></table>"
    )
    return HTML(html)


def paint_entities(text: str, entities: Sequence[Entity], ax_title: str) -> plt.Figure:
    kinds_order = ["TOPOLOGY", "COMPONENT", "PARAMETER", "CONSTRAINT", "VALUE", "UNIT"]
    colors = {
        "TOPOLOGY": PURPLE,
        "COMPONENT": ACCENT,
        "PARAMETER": GREEN,
        "CONSTRAINT": AMBER,
        "VALUE": ORANGE,
        "UNIT": RED,
    }
    y_index = {k: i for i, k in enumerate(kinds_order)}
    n = len(text)
    fig, ax = plt.subplots(figsize=(12, 3.0), dpi=120)
    ax.set_title(ax_title, color=FG, fontsize=12)
    for e in entities:
        y = y_index.get(e.kind)
        if y is None:
            continue
        ax.broken_barh([(e.start, e.end - e.start)], (y - 0.35, 0.7), facecolors=colors[e.kind], edgecolors="#30363d", linewidth=0.4)
    ax.set_yticks(list(range(len(kinds_order))))
    ax.set_yticklabels(kinds_order, color=MUTED, fontsize=10)
    ax.set_xlabel("Character index", color=MUTED)
    ax.set_xlim(0, max(n, 1))
    ax.set_ylim(-0.5, len(kinds_order) - 0.2)
    ax.grid(True, axis="x", linestyle="--", linewidth=0.6)
    for spine in ax.spines.values():
        spine.set_color("#30363d")
    fig.subplots_adjust(left=0.14, right=0.98, top=0.88, bottom=0.18)
    fig.text(0.14, 0.04, text[:200] + ("…" if len(text) > 200 else ""), color=MUTED, fontsize=9, va="bottom")
    return fig


def entity_kind_counts_figure(entities: Sequence[Entity], title: str) -> go.Figure:
    from collections import Counter

    c = Counter(e.kind for e in entities)
    order = ["TOPOLOGY", "COMPONENT", "PARAMETER", "CONSTRAINT", "VALUE", "UNIT"]
    xs = [c.get(k, 0) for k in order]
    fig = go.Figure(
        go.Bar(
            x=xs,
            y=order,
            orientation="h",
            marker_color=[PURPLE, ACCENT, GREEN, AMBER, ORANGE, RED],
        )
    )
    fig.update_layout(
        title=title,
        paper_bgcolor=DARK_BG,
        plot_bgcolor="#161b22",
        font=dict(color=FG, size=12),
        margin=dict(l=120, r=24, t=48, b=40),
        xaxis=dict(gridcolor="#30363d", title="Count"),
        yaxis=dict(showgrid=False),
        height=280,
    )
    return fig


# Build rows for side-by-side HTML
rows_html: List[Tuple[str, str, str]] = []
figs: List[plt.Figure] = []

for title, spec in EXAMPLES.items():
    inv = build_component_inventory(spec)
    inv["validation"] = validate_inventory(inv)
    rows_html.append((title, spec, inventory_to_compact_json(inv)))
    ner = HardwareNER()
    ents = ner.extract(preprocess_spec(spec))
    figs.append(paint_entities(preprocess_spec(spec), ents, ax_title=f"Entity spans — {title}"))

display(side_by_side_table(rows_html))
for f in figs:
    plt.show()

# Plotly (plotly_dark): entity histogram for the primary specification
_primary_ents = HardwareNER().extract(preprocess_spec(PRIMARY_SPEC))
display(entity_kind_counts_figure(_primary_ents, "Entity counts — OTA (primary)"))

print("\nFull primary inventory (`inv0`) keys:", list(inv0.keys()))
